# Analytical Evaluation Report: Audio Chord Recognition

This notebook serves as a publication-quality evaluation report for our **Transformer-based Audio Chord Recognition** model, prepared for defense presentation and academic review. 

We evaluate the model on two levels:
1. **Macro Evaluation**: Global performance across the entire validation dataset, presenting a detailed classification report (Precision, Recall, F1) and a normalized confusion matrix highlighting structural misclassifications.
2. **Micro Evaluation (Single-Track Deep Dive)**: A detailed temporal analysis of a single track chunk, aligning the CQT spectrogram, ground truth annotations, and model predictions, with dotted indicators showing transition boundary alignment.

In [ ]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import Audio

# Append path to import src modules
sys.path.append('../')
from src.dataset import get_dataloaders
from src.model import TransformerChordRecognizer

print("Setup complete. PyTorch version:", torch.__version__)
print("Seaborn version:", sns.__version__)

### Load Model and Validation Dataset

We instantiate the trained model, load weights from `best_model.pth`, and retrieve the validation data loader.

In [ ]:
import os
import sys
import torch

sys.path.append('../')
from src.dataset import get_dataloaders
from src.model import TransformerChordRecognizer

# Setup execution device
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Using device:", device)

# Initialize Model
input_bins = 84
num_classes = 25
model = TransformerChordRecognizer(input_bins=input_bins, num_classes=num_classes)

# Load weights
checkpoint_path = '../models/best_model.pth'
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Successfully loaded model weights from: {checkpoint_path}")
else: 
    print(f"Error: checkpoint not found at {checkpoint_path}", file=sys.stderr)

model.to(device)
model.eval()

# Load the ENTIRE Validation DataLoader
processed_dir = '../data/processed'
_, val_loader = get_dataloaders(processed_dir, batch_size=32, train_split=0.8)
print(f"Validation dataloader ready. Total validation batches: {len(val_loader)}")

### Macro Evaluation: Classification Report & Global Confusion Matrix

We iterate over all batches in the validation loader to compute global frame-level predictions. We then generate standard metrics (precision, recall, F1-score) and visualize class confusions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_targets = []

# Collect predictions over validation loader
with torch.no_grad():
    for X_val, y_val in val_loader:
        # Reshape features from [batch_size, 84, num_frames] to [batch_size, num_frames, 84]
        X_val_input = X_val.permute(0, 2, 1).to(device)
        y_val = y_val.to(device)
        
        logits = model(X_val_input)
        preds = logits.argmax(dim=2)
        
        all_preds.extend(preds.cpu().numpy().flatten())
        all_targets.extend(y_val.cpu().numpy().flatten())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Define labels for the 25 chord classes
chord_names = [
    'C:maj', 'C#:maj', 'D:maj', 'D#:maj', 'E:maj', 'F:maj', 'F#:maj', 'G:maj', 'G#:maj', 'A:maj', 'A#:maj', 'B:maj',
    'C:min', 'C#:min', 'D:min', 'D#:min', 'E:min', 'F:min', 'F#:min', 'G:min', 'G#:min', 'A:min', 'A#:min', 'B:min',
    'N'
]

# Generate classification report
print("=== Frame-level Classification Report ===\n")
print(classification_report(all_targets, all_preds, labels=np.arange(25), target_names=chord_names, zero_division=0))

# Compute confusion matrix
cm = confusion_matrix(all_targets, all_preds, labels=np.arange(25))
# Normalize by row (support)
row_sums = cm.sum(axis=1)[:, np.newaxis]
cm_norm = np.divide(cm.astype('float'), row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums!=0)

# Plotting the Confusion Matrix
plt.figure(figsize=(16, 13))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", cbar=True,
            xticklabels=chord_names, yticklabels=chord_names,
            cbar_kws={'label': 'Normalized Accuracy (Recall)'})
plt.title("Global Confusion Matrix for Frame-level Chord Recognition\n(Normalized by Class Support)", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Predicted Chord Class", fontsize=12, labelpad=10)
plt.ylabel("Ground Truth Chord Class", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

### Micro Evaluation: Single Track Temporal Alignment and Boundary Analysis

Here, we inspect a single chunk from the validation dataset. We perform inference, plot the input CQT Spectrogram (using note frequencies), and draw the corresponding Ground Truth and Prediction ribbons. 

Dotted vertical lines are plotted at each frame index where the Ground Truth chord changes. This boundary alignment enables us to see if the model has a lag or lead at chord transitions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa.display

# Fetch a single validation chunk using batch_size=1
_, single_val_loader = get_dataloaders(processed_dir, batch_size=1, train_split=0.8)
X_batch, y_batch = next(iter(single_val_loader))

# Run inference
X_batch_input = X_batch.permute(0, 2, 1).to(device)
with torch.no_grad():
    logits = model(X_batch_input)
    predictions = torch.argmax(logits, dim=2)

cqt_numpy = X_batch[0].cpu().numpy()
ground_truth = y_batch[0].cpu().numpy()
predictions_numpy = predictions[0].cpu().numpy()

# Parameters
sr = 22050
hop_length = 512
num_frames = cqt_numpy.shape[1]
duration = num_frames * hop_length / sr

# Find ground truth transition frame indices
transitions = []
for t in range(1, len(ground_truth)):
    if ground_truth[t] != ground_truth[t - 1]:
        transitions.append(t)
transition_times = [t * hop_length / sr for t in transitions]

# Create plot
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True, 
                         gridspec_kw={'height_ratios': [4, 1, 1]})

# 1. Plot CQT Spectrogram
fmin = librosa.note_to_hz('C1')
librosa.display.specshow(
    librosa.amplitude_to_db(cqt_numpy, ref=np.max),
    sr=sr,
    hop_length=hop_length,
    x_axis='time',
    y_axis='cqt_note',
    fmin=fmin,
    ax=axes[0],
    cmap='coolwarm'
)
axes[0].set_title("Input CQT Spectrogram (Spectro-Temporal Features)", fontsize=14, fontweight='bold', pad=10)
axes[0].set_ylabel("Pitch (Note Bins)", fontsize=12)

# Setup distinct colormap for chord classes
cmap_chords = plt.colormaps.get_cmap('hsv')

# 2. Plot Ground Truth Ribbon
axes[1].imshow(ground_truth[np.newaxis, :], aspect='auto', cmap=cmap_chords, extent=[0, duration, 0, 1], vmin=0, vmax=24)
axes[1].set_title("Ground Truth Chord Alignments", fontsize=12, fontweight='bold', pad=5)
axes[1].set_yticks([])
axes[1].set_ylabel("GT Chords", fontsize=12)

# 3. Plot Predictions Ribbon
pred_img = axes[2].imshow(predictions_numpy[np.newaxis, :], aspect='auto', cmap=cmap_chords, extent=[0, duration, 0, 1], vmin=0, vmax=24)
axes[2].set_title("Model Predicted Chord Alignments", fontsize=12, fontweight='bold', pad=5)
axes[2].set_yticks([])
axes[2].set_ylabel("Predicted", fontsize=12)
axes[2].set_xlabel("Time (Seconds)", fontsize=12)

# Plot vertical dotted line markers at Ground Truth chord boundaries
for t_time in transition_times:
    for ax in axes:
        ax.axvline(x=t_time, color='black', linestyle=':', linewidth=1.5, alpha=0.85)

# Format Legend Colorbar
cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.02])
cbar = fig.colorbar(pred_img, cbar_ax, orientation='horizontal')
cbar.set_ticks(np.arange(0, 25))
cbar.set_ticklabels(chord_names)
cbar.ax.tick_params(labelsize=8, rotation=45)

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

# Display matching metric
correct = np.sum(ground_truth == predictions_numpy)
total = len(ground_truth)
accuracy = (correct / total) * 100
print(f"Sequence Length: {total} frames")
print(f"Matching accuracy for this specific sequence: {accuracy:.2f}%")

### Interactive Audio Playback

**Defense Presentation Note**:
During thesis defense presentations, visual analysis is greatly enhanced by direct auditory verification.
Since HDF5 files contain pre-processed CQT matrices, you should load the original source audio `.mp3`/`.wav` file corresponding to this song segment here.

Run the cell below with the target file path to enable an interactive playback bar under the plots:

In [ ]:
from IPython.display import Audio

# Replace with the actual absolute path to the audio file for the presentation
audio_path = '../data/raw/McGill-Billboard/audio.wav' 

if os.path.exists(audio_path):
    display(Audio(filename=audio_path))
else:
    print(f"Placeholder: Place the audio file at '{audio_path}' to enable defense playback.")